## TUGAS


**Nama : Cahyo Adi Nugroho**

**NPM : 2505060034**

In [1]:
#Membuat data set

import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [2]:
#memulai sparksession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pandas as pd

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas5_BigData") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

26/09/17 20:47:17 WARN Utils: Your hostname, cahyoalim-ThinkPad-L14-Gen-2 resolves to a loopback address: 127.0.1.1; using 192.168.1.28 instead (on interface wlp9s0)
26/09/17 20:47:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 20:47:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/17 20:47:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [11]:
#membuat tabel transaksi dari csv yang sudah diupload ke hdfs
from pyspark.sql.functions import col

transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True) 

transaksi = transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

print("Data Transaksi")
transaksi.show()

Data Transaksi
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
|   TRX-5|Kesehatan & Kecan...|  Magelang|           4|      100000|    400000|
|   TRX-6|        Rumah Tangga|  Magelang|           6|       25000|    150000|
|   TRX-7|          Elektronik|  Semarang|           8|       50000|    400000|
|   TRX-8|             Fashion|  Semarang|           5|       25000|    125000|
|   TRX-9|          Elekt

In [15]:
#Mengubah   dictionary data_target_cabaang menjadi tabel (dataframe)
target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



In [27]:
#A.  Join & Perbandingan Target 

from pyspark.sql.functions import sum as spark_sum, col

#mengelomopkkan total pendapatan per kota
kota = transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))

#menggabungkan dengan tabel target
hasil = kota.join(target, on="kota", how="inner")

#menambahkan kolom pencapaian_persen
hasil = hasil.withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100)


#mengurutkan pencapaian_persen dari yang tertinggi
hasil = hasil.orderBy(col("pencapaian_persen").desc())

hasil.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [32]:
#B. Window Function — Kategori Terlaris per Kota
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum as spark_sum, row_number


pengelompokan = transaksi.groupBy("kota", "kategori").agg(spark_sum("pendapatan").alias("total_pendapatan"))

window = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

peringkat = pengelompokan.withColumn("rank", row_number().over(window))

terlaris = peringkat.filter(col("rank")  == 1).drop("rank")

terlaris.show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|  Magelang|Kesehatan & Kecan...|         7275000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|  Semarang|        Rumah Tangga|        11125000|
|      Solo|Kesehatan & Kecan...|         8425000|
|Yogyakarta|             Fashion|        13325000|
+----------+--------------------+----------------+



In [36]:
#C. Spark SQL

#mendaftarkan tabel transaksi dan tabel target menjadi temporary view
transaksi.createOrReplaceTempView("transaksi")
target.createOrReplaceTempView("target")

hasil = spark.sql('''SELECT transaksi.kota, target.pic_cabang, COUNT(transaksi.order_id) AS jumlah_transaksi
FROM transaksi JOIN target on transaksi.kota = target.kota 
GROUP BY transaksi.kota, target.pic_cabang 
ORDER BY  jumlah_transaksi DESC           ''')

hasil.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**kesimpulan**

Berdasarkan analisis kombinasi pada Bagian A dan B, cabang yang menunjukkan kinerja paling baik adalah Cabang pada kota Purworejo di bawah kepemimpinan Fitri. Cabang ini berhasil menghasilkan total pendapatan sebesar Rp45.650.000 dari target bulanan Rp30.000.000. Angka tersebut menandakan pencapaian sebesar 152,17%, menjadikannya satu-satunya cabang yang sukses melampaui target. Sebaliknya, cabang yang paling membutuhkan perhatian mendalam dari manajemen adalah Cabang di kota Semarang yang dikelola oleh Sari. Meskipun mampu mencatatkan penghasilan sebesar Rp38.175.000, persentase pencapaiannya berada di urutan paling bawah, yaitu hanya 69,41% dari target sebesar Rp55.000.000. Selain Semarang, Cabang kota Magelang juga masih tergolong rendah dengan tingkat ketercapaian 70,33% (Rp31.650.000 dari target Rp45.000.000).



In [37]:
spark.stop()